In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
import os

def show_tree(root, max_depth=3, max_items_per_dir=10):
    root = os.path.abspath(root)
    print(root)
    for current_root, dirs, files in os.walk(root):
        rel = os.path.relpath(current_root, root)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "    " * depth
        if rel != ".":
            print(f"{indent}|-- {os.path.basename(current_root)}/")

        shown_dirs = sorted(dirs)[:max_items_per_dir]
        shown_files = sorted(files)[:max_items_per_dir]

        for d in shown_dirs:
            print(f"{indent}    |-- {d}/")
        for f in shown_files:
            print(f"{indent}    |-- {f}")

        if len(dirs) > max_items_per_dir or len(files) > max_items_per_dir:
            print(f"{indent}    |-- ...")

show_tree("/kaggle/input/datasets/faariskhairrudin/", max_depth=3)

In [ ]:
!pip install ultralytics -q

In [ ]:
!ls /kaggle/input/datasets/faariskhairrudin/rdd2024-6-classes/RDD_Split_Final_china_japan_india_6_classes/

In [ ]:
import yaml
import os

original_yaml_path = '/kaggle/input/datasets/faariskhairrudin/rdd2024-6-classes/RDD_Split_Final_china_japan_india_6_classes/data.yaml'

with open(original_yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = '/kaggle/input/datasets/faariskhairrudin/rdd2024-6-classes/RDD_Split_Final_china_japan_india_6_classes/'
data['train'] = 'train/images'
data['val'] = 'val/images'

new_yaml_path = '/kaggle/working/kaggle_data.yaml'
with open(new_yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f"File YAML baru berhasil dibuat di: {new_yaml_path}")

In [ ]:
from ultralytics import YOLO

# Skenario 1: Memuat model YOLOv8n (Baseline)
model = YOLO('yolov8n.pt')

results = model.train(
    data='/kaggle/working/kaggle_data.yaml',
    epochs=300,
    imgsz=640,           # Sesuai teks Bab 3: resolusi citra 640x640
    batch=128,            # Aman untuk 2 GPU Kaggle (jika memori sisa banyak, bisa dicoba 128)
    device=[0, 1],       # MENGGUNAKAN 2 GPU KAGGLE SEKALIGUS agar 2x lebih cepat
    workers=4,
    cache=False,
    name='RDD_Baseline',
    project='RDD_Project',
    deterministic=True,  # Menjaga reproduksibilitas eksperimen
    half=True,           # FP16 Precision (Mempercepat training di Kaggle T4)

    # --- PARAMETER PENYESUAIAN METODOLOGI PROPOSAL ---
    mosaic=1.0,
    # fl_gamma=1.5
)

In [ ]:
import shutil
import os
from datetime import datetime

# Path hasil training
source_folder = '/kaggle/working/runs/detect/RDD_Project/RDD_Baseline-3'
zip_file = '/kaggle/working/RDD_Baseline_Final.zip'

# Kompres
shutil.make_archive('/kaggle/working/RDD_Baseline_Final', 'zip', source_folder)

# Upload sebagai Kaggle Dataset
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
dataset_name = f'rdd-baseline-{timestamp}'

os.system(f'kaggle datasets create -p /kaggle/working -r zip --quiet -n "{dataset_name}" -s "RDD Baseline Model"')

print(f"✅ Dataset '{dataset_name}' berhasil di-upload ke Kaggle!")
print("📥 Model tersimpan permanent di akun Kaggle kamu")

In [ ]:
!kaggle kernels output faariskhairrudin/nrdd2024-model -p ./downloaded_model

In [ ]:
# Copy ke workspace folder yang persistent
import shutil
persistent_path = '/root/.kaggle/models/'
shutil.copy(zip_file, persistent_path)

In [ ]:
import shutil
import os

# Path folder yang mau diambil
folder_path = '/kaggle/working/runs/detect/RDD_Project/RDD_Baseline4'
# Nama file zip hasil kompresi
output_filename = '/kaggle/working/RDD_Baseline4_Final'

# Proses kompresi
shutil.make_archive(output_filename, 'zip', folder_path)

print(f"✅ Folder berhasil di-zip ke: {output_filename}.zip")

In [ ]:
import torch

# Hapus cache GPU
torch.cuda.empty_cache()
print("✅ GPU cache berhasil dihapus!")
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")